# Session 2.1 : Data Transformation

_Analytics Through Coding Autumn 2026_

---

Visualisation and statistical analysis are only useful if the data are in the form required to answer the question.

In practice, we often need to:

* keep only relevant observations;
* reorder observations;
* select or rename variables;
* create new variables;
* group observations;
* calculate summaries.

In this session we will use the `flights` dataset, which contains flights that departed from New York City in 2013.

Rather than learning pandas functions in isolation, we will start with an **analytical question** and then decide what transformation is needed.

---

## Starting out

As always, import the necessary libraries and dataset at the start of the notebook.

In [1]:
import pandas as pd
import numpy as np

flights = pd.read_csv("../Data/nycflights13_flights.csv", index_col=0)
flights.reset_index(drop=True, inplace=True)

flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00


Before transforming a dataset, remind yourself what one row represents and check the variables available.

In [2]:
print("Shape:", flights.shape)
print(flights.columns.tolist())

Shape: (202066, 19)
['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight', 'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute', 'time_hour']


## A small set of transformation tools

We will focus on a small number of pandas methods that can be combined to answer many analytical questions.

| pandas method | What it helps us do |
|---|---|
| `query()` | Keep observations that satisfy a condition |
| `sort_values()` | Reorder observations |
| `loc[]` | Select rows and/or columns |
| `rename()` | Rename variables |
| `assign()` | Create new variables |
| `groupby()` | Divide observations into groups |
| `agg()` | Calculate summaries |

The important skill is not memorising this table. It is recognising **which operation is required by the analytical question**.

## Question 1: Which flights departed on 16 August?

We only want observations where:

* `month == 8`
* `day == 16`

We can use `.query()` to filter observations.

In [3]:
flights_aug16 = flights.query('month == 8 & day == 16')
flights_aug16.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
286,2013,8,16,1510.0,1455,15.0,1652.0,1701,-9.0,9E,4120,N8775A,JFK,CLE,73.0,425,14,55,2013-08-16 14:00:00
410,2013,8,16,1256.0,1255,1.0,1543.0,1545,-2.0,UA,1641,N19136,EWR,MCO,143.0,937,12,55,2013-08-16 12:00:00
707,2013,8,16,1959.0,2000,-1.0,2242.0,2310,-28.0,DL,2391,N916DL,JFK,TPA,146.0,1005,20,0,2013-08-16 20:00:00
1564,2013,8,16,619.0,620,-1.0,906.0,843,23.0,DL,1743,N6704Z,JFK,ATL,121.0,760,6,20,2013-08-16 06:00:00
1573,2013,8,16,2156.0,2159,-3.0,2258.0,2324,-26.0,UA,1116,N71411,EWR,BOS,42.0,200,21,59,2013-08-16 21:00:00


Notice that pandas returns a **new DataFrame**. The original `flights` DataFrame has not been changed.

In [4]:
print("Shape:", flights.shape)
print("Shape:", flights_aug16.shape)

Shape: (202066, 19)
Shape: (608, 19)


Multiple arguments to `.query()` are combined with `“and”`: every expression must be true in order for a row to be included in the output. For some operations you may need other Boolean operations - `&` is “and”, `|` is “or”, and `!` is “not”

![GitHub Codespaces](Boolean_operators.png)

### Exercise 1

Find all flights that:

* departed from `JFK`;
* travelled to `LAX`; and
* had a departure delay greater than 60 minutes.

Keep the result in a DataFrame called `jfk_lax_delayed`.

How many flights meet all three conditions?

In [10]:
jfk_lax_delayed = flights.query("origin == 'JFK' & dest == 'LAX' & dep_delay > 60")

jfk_lax_delayed.head()
jfk_lax_delayed.shape

(359, 19)

## Question 2: Which flights experienced the largest departure delays?

Filtering determines **which observations we keep**.

Sorting determines **the order in which we inspect them**.

In [16]:
flights.sort_values('dep_delay', ascending=False)[['month', 'day', 'origin', 'dest']].head()

,month,day,origin,dest
114519,1,9,JFK,HNL
74463,1,10,EWR,ORD
176637,9,20,JFK,SFO
201196,3,17,LGA,MSP
53590,7,22,LGA,ATL


We can also sort using more than one variable.

For example, the following sorts chronologically by month and day.

In [15]:
flights.sort_values(
    by = ['month', 'day', 'dep_time']
)[['month', 'day']].head()

,month,day
49463,1,1
168799,1,1
8721,1,1
114668,1,1
58344,1,1


### Exercise 2

Find the **10 flights with the longest arrival delays**.

Display only:

* `origin`
* `dest`
* `carrier`
* `arr_delay`

Sort the result from the largest delay to the smallest.

In [ ]:
##option 1
flights.sort_values('arr_delay', ascending=False)[
    ['origin', 'dest', 'carrier', 'arr_delay']
].head(10)


,origin,dest,carrier,arr_delay
114519,JFK,HNL,HA,1272.0
74463,EWR,ORD,MQ,1109.0
176637,JFK,SFO,AA,1007.0
201196,LGA,MSP,DL,915.0
53590,LGA,ATL,DL,895.0
150167,EWR,ORD,MQ,875.0
194528,JFK,TPA,DL,856.0
54279,JFK,LAS,AA,852.0
103118,JFK,BWI,MQ,851.0
151237,EWR,SLC,DL,847.0


In [ ]:
##option 2
flights.loc[:, ['origin', 'dest', 'carrier', 'arr_delay']] \
.sort_values('arr_delay', ascending=False)\
.head(10)

,origin,dest,carrier,arr_delay
114519,JFK,HNL,HA,1272.0
74463,EWR,ORD,MQ,1109.0
176637,JFK,SFO,AA,1007.0
201196,LGA,MSP,DL,915.0
53590,LGA,ATL,DL,895.0
150167,EWR,ORD,MQ,875.0
194528,JFK,TPA,DL,856.0
54279,JFK,LAS,AA,852.0
103118,JFK,BWI,MQ,851.0
151237,EWR,SLC,DL,847.0


## Question 3: Which variables do we actually need?

Real datasets often contain many more variables than are required for a particular question.

For an analysis of flight delays, we might only need a subset of columns.

In [28]:
delay_variables = flights.loc[
    :,
    ['month', 'day', 'carrier', 'origin', 'dest', 'dep_delay', 'arr_delay']
]

delay_variables.head()

,month,day,carrier,origin,dest,dep_delay,arr_delay
0,3,25,UA,EWR,RSW,24.0,19.0
1,4,26,DL,JFK,SFO,-4.0,-37.0
2,5,21,EV,EWR,DCA,11.0,16.0
3,7,18,EV,EWR,CLT,-8.0,-22.0
4,8,29,B6,JFK,BQN,-5.0,0.0


We can also rename variables when a clearer name would make later code easier to read.

In [29]:
delay_variables.rename(
    columns = {
        'dep_delay': 'departure_delay',
        'arr_delay': 'arrival_delay'
    }
)

,month,day,carrier,origin,dest,departure_delay,arrival_delay
0,3,25,UA,EWR,RSW,24.0,19.0
1,4,26,DL,JFK,SFO,-4.0,-37.0
2,5,21,EV,EWR,DCA,11.0,16.0
3,7,18,EV,EWR,CLT,-8.0,-22.0
4,8,29,B6,JFK,BQN,-5.0,0.0
...,...,...,...,...,...,...,...
202061,1,27,US,EWR,CLT,-4.0,-13.0
202062,8,8,DL,LGA,ATL,225.0,209.0
202063,1,30,B6,JFK,FLL,73.0,73.0
202064,3,18,DL,LGA,ATL,-3.0,8.0


<div class="alert alert-warning">
<b>Note.</b>
Selecting or renaming columns does not improve the analysis by itself. Do it when it makes the dataset easier to understand or when only a smaller set of variables is needed for the question.
</div>

## Question 4: Did flights make up time while in the air?

Sometimes the variable we need does not exist in the original dataset.

We can create a new variable from existing variables using `.assign()`.

Define:

`gain = arrival delay - departure delay`

A negative value means the flight arrived with **less delay** than it had when it departed.

In [30]:
flights_with_gain = flights.assign(
    gain = flights['arr_delay'] - flights['dep_delay']
)

flights_with_gain.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,gain
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,-5.0
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,-33.0
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,5.0
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,-14.0
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,5.0


We can create several new variables at the same time.

For example, approximate average speed in miles per hour can be calculated from `distance` and `air_time`.

In [31]:
flights_transformed = flights.assign(
    gain = flights['arr_delay'] - flights['dep_delay'],
    speed = flights['distance'] / (flights['air_time'] / 60)
)

flights_transformed.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,gain,speed
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,...,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,-5.0,379.171598
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,...,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,-33.0,460.415430
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,...,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,5.0,306.153846
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,...,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,-14.0,412.207792
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,...,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,5.0,477.575758


### Exercise 3

Create a DataFrame called `flights_delay_change` containing a new variable:

`delay_change = arr_delay - dep_delay`

Then keep only flights where `delay_change <= -30`.

These are flights that reduced their delay by at least 30 minutes between departure and arrival.

Display the 10 flights with the largest reduction in delay.

In [ ]:
flights_delay_change = (
    flights.assign(
    delay_change = flights['arr_delay'] - flights['dep_delay'],
)
.query('delay_change <= -30')
)


flights_transformed.head(10)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,gain,speed
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,...,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,-5.0,379.171598
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,...,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,-33.0,460.415430
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,...,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,5.0,306.153846
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,...,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,-14.0,412.207792
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,...,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,5.0,477.575758
5,2013,11,14,2031.0,2025,6.0,2342.0,2340,2.0,UA,...,N518UA,JFK,LAX,342.0,2475,20,25,2013-11-14 20:00:00,-4.0,434.210526
6,2013,3,30,1755.0,1755,0.0,2110.0,2114,-4.0,B6,...,N646JB,JFK,LAX,338.0,2475,17,55,2013-03-30 17:00:00,-4.0,439.349112
7,2013,8,10,1544.0,1545,-1.0,2003.0,2001,2.0,DL,...,N3764D,JFK,SJU,205.0,1598,15,45,2013-08-10 15:00:00,3.0,467.707317
8,2013,3,27,1855.0,1900,-5.0,2019.0,2028,-9.0,9E,...,N8623A,JFK,BWI,37.0,184,19,0,2013-03-27 19:00:00,-4.0,298.378378
9,2013,5,13,1912.0,1915,-3.0,2055.0,2154,-59.0,DL,...,N3736C,JFK,PHX,262.0,2153,19,15,2013-05-13 19:00:00,-56.0,493.053435


## Question 5: What is a typical delay?

`.agg()` allows us to collapse many observations into summary statistics.

Without grouping, the summary describes the **entire dataset**.

In [36]:
flights.agg(
    mean_departure_delay =  ('dep_delay', 'mean'),
    median_departure_delay = ('dep_delay', 'median'),
    mean_arrival_delay = ('arr_delay', 'mean')
)

,dep_delay,arr_delay
mean_departure_delay,12.623232,NaN
median_departure_delay,-2.000000,NaN
mean_arrival_delay,NaN,6.944759


## Question 6: Does delay differ between airports or airlines?

Usually we want summaries **within groups**.

`groupby()` changes the unit of analysis.

Instead of one row representing one flight, the resulting table can have one row representing one airport, airline, month, destination, or another group.

In [38]:
delay_by_origin = (
    flights.groupby('origin')
    .agg(
        flights = ('flight', 'size'),
        mean_departure_delay = ('dep_delay', 'mean'),
        mean_arrival_delay = ('arr_delay', 'mean')

    )
)

delay_by_origin.head(10)

,flights,mean_departure_delay,mean_arrival_delay
origin,,,
EWR,72435,15.083630,9.136757
JFK,66846,12.109672,5.638486
LGA,62785,10.328891,5.817045


This is an important conceptual change:

> Before `groupby()`: one row = one flight  
> After `groupby()` + `agg()`: one row = one origin airport

Always know what **one row represents** after a transformation.

### Exercise 4

Calculate the following for each airline (`carrier`):

* `n_flights`: number of flights;
* `avg_dep_delay`: average departure delay;
* `avg_arr_delay`: average arrival delay.

Keep only airlines with at least 1,000 flights and sort them from the **lowest to highest average arrival delay**.

In [ ]:
##puedo usar size para contar total o count para no contar nulos
delay_by_origin_2 = (
    flights.groupby('carrier')
    .agg(
        flights = ('flight', 'count'),
        mean_departure_delay = ('dep_delay', 'mean'),
        mean_arrival_delay = ('arr_delay', 'mean')

    )
    .query('flights >= 1000')
    .sort_values('mean_arrival_delay', ascending=False)
)

delay_by_origin_2.head(10)

,flights,mean_departure_delay,mean_arrival_delay
carrier,,,
FL,1962,18.042144,19.608877
EV,32519,19.976493,15.858449
MQ,15806,10.901307,10.991669
B6,32910,13.185801,9.764146
WN,7394,17.047298,9.123862
9E,11032,16.512983,7.021901
UA,35205,11.869524,3.323555
US,12384,3.866450,2.287291
DL,28774,9.363554,1.848063


## Combining transformations

Real analytical questions usually require more than one operation.

Method chaining lets us read the analysis as a sequence:

1. start with the data;
2. create or modify variables;
3. group observations;
4. calculate summaries;
5. filter the summaries;
6. order the result.

This mirrors the analytical workflow more closely than learning each method separately.

In [46]:
delay_by_origin_2 = (
    flights.groupby('carrier')
    .agg(
        flights = ('flight', 'count'),
        mean_departure_delay = ('dep_delay', 'mean'),
        mean_arrival_delay = ('arr_delay', 'mean')

    )
    .query('flights >= 1000 & mean_departure_delay > 5')
    .sort_values('mean_arrival_delay', ascending=False)
)

delay_by_origin_2.head(10)

,flights,mean_departure_delay,mean_arrival_delay
carrier,,,
FL,1962,18.042144,19.608877
EV,32519,19.976493,15.858449
MQ,15806,10.901307,10.991669
B6,32910,13.185801,9.764146
WN,7394,17.047298,9.123862
9E,11032,16.512983,7.021901
UA,35205,11.869524,3.323555
DL,28774,9.363554,1.848063
VX,3109,12.668714,1.578196


### Final Exercise — Average speed by destination

Imagine you want to know how fast flights travel on average depending on destination.

Create a new variable:

`speed = distance / (air_time / 60)`

Then, for each destination (`dest`), calculate:

* `count`: number of flights;
* `avg_speed`: average speed in miles per hour.

Keep only destinations with **more than 20 flights** and sort them by `avg_speed` from fastest to slowest.

Display the 10 fastest destinations.

Try to write this as a single method chain.

In [51]:

delay_by_destination = (
    flights
    .assign(
        speed = flights['distance'] / (flights['air_time'] / 60)
)
    .groupby('dest', dropna = True)
    .agg(
        flights = ('flight', 'count'),
        avg_speed = ('speed', 'mean')

    )
    .query('flights > 20')
    .reset_index()
    .sort_values('avg_speed', ascending=False)
)

delay_by_destination.head(10)

,dest,flights,avg_speed
11,BQN,547,486.941156
83,SJU,3521,485.500289
37,HNL,428,483.518542
69,PSE,229,481.060702
89,STT,289,478.681312
45,LAX,9687,452.985962
85,SMF,164,451.506080
76,SAN,1647,451.474604
46,LGB,390,449.623567
81,SFO,8027,448.619269


## Check the result

A transformation is not finished just because the code ran.

Before using the result, ask:

* Does one row now represent what I think it represents?
* Are the number of groups plausible?
* Are the summary values plausible?
* Did missing values affect the calculation?
* Did filtering happen before or after aggregation as intended?

A few simple checks can prevent incorrect conclusions.

## All Done!

In this session we used pandas transformations to answer analytical questions.

We practised:

* filtering observations with `query()`;
* sorting observations with `sort_values()`;
* selecting variables with `loc[]`;
* renaming variables;
* creating new variables with `assign()`;
* grouping observations with `groupby()`;
* calculating summaries with `agg()`;
* combining several operations using method chaining;
* checking that transformed results still make analytical sense.

The key idea is:

> **Start with the question, then decide what data transformation is required.**

We will now move on to the structure of datasets and the principles of **tidy data**.